This notebook provides end-to-end examples for loading LegalPincite, running retrieval, reranking results with cross-encoder (MonoT5) and bi-encoder (LegalBERT).

1. Loading Dataset
    - With `datasets` library
    - With `pandas` library
2. Load or Create Index
    - Load PyTerrier index from HuggingFace
    - Create index with PyTerrier
3. Simple sparse retrieval (with BM25) and evaluation
    - With own queries (not from the dataset)
    - With a single query from the dataset
    - Simple evaluation
4. Full experimental pipeline and evaluation
    - First-stage (sparse) retrieval
    - Reranking with MonoT5 (cross-encoder)
    - Reranking with LegalBERT (bi-encoder)

In [1]:
import warnings
warnings.filterwarnings("all")

In [2]:
# !pip install pyterrier[java]

In [3]:
import pyterrier as pt

# === Needed to resolve "Unable to find JAVA_HOME" error

def set_up_pt():
    java_home = "C:/Program Files/Java/jdk-25.0.2" # change to your own path to Java
    pt.java.set_java_home(java_home)

set_up_pt() 

# 1. Loading dataset

We demonstrate two ways of loading `LegalPincite`: with HuggingFace's `datasets` library and with the `pandas` library.

## With `datasets` library

In [4]:
from datasets import load_dataset
from pprint import pp

# Example config to load dataset for paragraph-to-paragraph retrieval (test split)
query_config = "query_par"
doc_config = "doc_par"
qrel_config = "qrel_par_par"
split = "test"

queries = load_dataset("theresiavr/legalpincite", query_config, split=split)
docs = load_dataset("theresiavr/legalpincite", doc_config, split=split)
qrels = load_dataset("theresiavr/legalpincite", qrel_config, split=split)

pp(queries[0])
pp(docs[0])
print(qrels[0])

{'qid': '62018CJ0529-21',
 'query_unmasked': '21 In paragraph 54 of the order under appeal, the General '
                   'Court recalled, on the basis of the judgment of 6 '
                   'September 2012, Prezes Urzędu Komunikacji Elektronicznej v '
                   'Commission (C‑422/11 P and C‑423/11 P, EU:C:2012:553, '
                   'paragraph 24 and the case-law cited), that the concept of '
                   '‘independence’ of lawyers is defined not only positively, '
                   'namely by reference to professional ethical obligations, '
                   'but also negatively, that is to say, by the absence of an '
                   'employment relationship between the lawyer and his or her '
                   'client. In paragraph 55 of the order under appeal, the '
                   'General Court held that that reasoning applied with the '
                   'same force in a situation in which a lawyer is employed by '
                   'an entity 

In [5]:
# Convert HF dataset structure to pandas dataframe, otherwise load dataset with pandas library
df_queries = queries.to_pandas()
df_docs = docs.to_pandas()
df_qrels = qrels.to_pandas()

## With ``pandas`` library

In [6]:
import pandas as pd

# Example script to load datasets for paragraph-to-paragraph retrieval (test split)
# Note: might take a minute

base = "https://huggingface.co/datasets/theresiavr/legalpincite/resolve/main"

df_queries = pd.read_csv(f"{base}/query_test_par.csv")
df_docs = pd.read_csv(f"{base}/doc_test_par.csv")
df_qrels = pd.read_csv(f"{base}/qrel_test_par_par.csv")

print(df_queries.head())
print()
print(df_docs.head())
print()
print(df_qrels.head())

              qid                                     query_unmasked  \
0  62018CJ0529-21  21 In paragraph 54 of the order under appeal, ...   
1  62018CJ0529-55  55 As a preliminary point, as regards EUIPO’s ...   
2  62018CJ0529-58  58 As to the substance, it should be recalled,...   
3  62018CJ0529-60  60 As regards the first condition, relating to...   
4  62018CJ0529-61  61 In that regard, and as the General Court no...   

                                               query  
0  21 In of the order under appeal, the General C...  
1  55 As a preliminary point, as regards ’s argum...  
2  58 As to the substance, it should be recalled,...  
3  60 As regards the first condition, relating to...  
4  61 In that regard, and as the General Court no...  

           docno                                               text
0  61954CJ0001-1  1 - ADMISSIBILITY THE PARTIES RAISE NO OBJECTI...
1  61954CJ0001-2  2 - THE SUBSTANCE OF THE CASE THE APPLICANT CO...
2  61954CJ0002-1  1 . SUR LA REC

# 2. Load or Create Index

An index is needed for efficient retrieval across documents (i.e., cases/paragraphs for LegalPincite). 

This part requires Java installation. See PyTerrier's guide on installing Java and the required version: <https://pyterrier.readthedocs.io/en/latest/troubleshooting/java.html#install-java>

## Load PyTerrier index from HuggingFace

In [7]:
# Load the index for paragraphs (test split)
index = pt.Artifact.from_hf('theresiavr/legalpincite_doc_test_par.terrier')

##  Create index with PyTerrier

In [10]:
level = "par"

# Create and save index of documents from the HF Dataset object
pt.terrier.TerrierIndex(f"index/doc_{split}_{level}.terrier")\
                                                        .indexer()\
                                                        .index(docs)

# Took ~34 minutes

Java started (triggered by TerrierIndexer.__init__) and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


# 3. Simple sparse retrieval (with BM25) and evaluation

In [8]:
bm25 = index.bm25(num_results=10) # use the loaded index from HF

Java started (triggered by Retriever.__init__) and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


## With own queries (not from the dataset)

In [9]:
result = bm25.search("human rights")
result

,qid,docid,docno,rank,score,query
0,1,496319,62017CJ0234-30,0,15.596180,human rights
1,1,331068,62010CJ0292-58,1,15.364356,human rights
2,1,484607,62016CJ0638-3,2,15.300392,human rights
3,1,374964,62012CJ0280-46,3,15.118097,human rights
4,1,447864,62015CJ0404-6,4,15.078549,human rights
5,1,391195,62013CJ0176-44,5,15.057795,human rights
6,1,440795,62015CJ0205-55,6,15.051189,human rights
7,1,284692,62008CJ0045-41,7,15.024667,human rights
8,1,391978,62013CJ0200-42,8,14.997986,human rights
9,1,540627,62018CJ0607-255,9,14.865890,human rights


In [10]:
# to get the text, merge with the pandas-loaded docs
result_with_text = result.merge(df_docs, left_on="docno", right_on="docno", how="left")
result_with_text

,qid,docid,docno,rank,score,query,text
0,1,496319,62017CJ0234-30,0,15.596180,human rights,"30 The referring court states, however, that i..."
1,1,331068,62010CJ0292-58,1,15.364356,human rights,"58. It is apparent, furthermore, from the case..."
2,1,484607,62016CJ0638-3,2,15.300392,human rights,3 Article 1 of the European Convention for the...
3,1,374964,62012CJ0280-46,3,15.118097,human rights,"46 The Council notes moreover that, according ..."
4,1,447864,62015CJ0404-6,4,15.078549,human rights,6 Article 1 of the Charter of Fundamental Righ...
5,1,391195,62013CJ0176-44,5,15.057795,human rights,44 It relies on Article 34 of the Convention f...
6,1,440795,62015CJ0205-55,6,15.051189,human rights,55 Such an interpretation of Article 47 of the...
7,1,284692,62008CJ0045-41,7,15.024667,human rights,41. It is also apparent from the Court’s case‑...
8,1,391978,62013CJ0200-42,8,14.997986,human rights,42 It relies on Article 34 of the European Con...
9,1,540627,62018CJ0607-255,9,14.865890,human rights,"255 Further, the right of access to evidence a..."


`score` above refers to the BM25 score. `rank` is based on that score, sorted from highest to lowest score.

## With a single query from the dataset

In [11]:
sample_query = df_queries.iloc[5]
sample_query_text = sample_query["query"]
pp(sample_query_text)

('62 That finding is confirmed by the context of the third paragraph of of the '
 'Statute of the Court of Justice of the European Union, from which it is '
 'clear that representation in legal proceedings of a party not covered by the '
 'first two paragraphs of that n be provided only by a lawyer, whereas the '
 'parties covered by those first two y be represented by an agent who may, '
 'where appropriate, be assisted by adviser or lawyer (see, to that effect, '
 'judgment of .')


In [12]:
sample_result = bm25.search(sample_query_text, qid=sample_query["qid"])
sample_result

,qid,docid,docno,rank,score,query
0,62018CJ0529-62,507204,62017CJ0515-60,0,77.448379,62 That finding is confirmed by the context of...
1,62018CJ0529-62,357245,62011CJ0422-2,1,59.773919,62 That finding is confirmed by the context of...
2,62018CJ0529-62,507146,62017CJ0515-2,2,55.032701,62 That finding is confirmed by the context of...
3,62018CJ0529-62,507199,62017CJ0515-55,3,42.493888,62 That finding is confirmed by the context of...
4,62018CJ0529-62,507156,62017CJ0515-12,4,40.527215,62 That finding is confirmed by the context of...
5,62018CJ0529-62,51723,61985CJ0427-9,5,39.907580,62 That finding is confirmed by the context of...
6,62018CJ0529-62,507186,62017CJ0515-42,6,39.054409,62 That finding is confirmed by the context of...
7,62018CJ0529-62,357283,62011CJ0422-40,7,37.164254,62 That finding is confirmed by the context of...
8,62018CJ0529-62,357276,62011CJ0422-33,8,35.772118,62 That finding is confirmed by the context of...
9,62018CJ0529-62,507201,62017CJ0515-57,9,35.441511,62 That finding is confirmed by the context of...


In [13]:
selected_qrel = df_qrels[df_qrels.qid.isin(sample_result.qid)]
selected_qrel # we see from the cell above that the relevant document is at rank 0 (top)

,qid,docno,label,source
6,62018CJ0529-62,62017CJ0515-60,1,eur_lex


## Simple evaluation

In [14]:
pt.Evaluate(sample_result, selected_qrel, metrics=["success", "recip_rank"])

{'Success@1': 1.0, 'Success@5': 1.0, 'Success@10': 1.0, 'recip_rank': 1.0}

# 4. Full experimental pipeline and evaluation

We provide example of a full experimental pipeline + evaluation to search for relevant paragraphs to be cited in the query paragraphs, using the `dev` split and only the ones that are validated by two human legal experts.

Additionally, we provide the code for the baseline experiments across all possible query-document level pairings (in the dataset paper) under the `script` folder of this repository.

## First-stage (sparse) retrieval

In [15]:
import os

save_folder = "experiment/example_human"

if not os.path.exists(save_folder):
    os.makedirs(save_folder)

In [16]:
# load query, qrel, and index

from datasets import load_dataset
import pyterrier as pt

split = "dev"

query_config = "query_par" #query-level
qrel_config = "qrel_par_par"
doc = "par" #doc-level

queries = load_dataset("theresiavr/legalpincite", query_config, split=split)
qrels = load_dataset("theresiavr/legalpincite", qrel_config, split=split)

df_queries = queries.to_pandas()
df_qrels = qrels.to_pandas()

index = pt.Artifact.from_hf(f'theresiavr/legalpincite_doc_{split}_{doc}.terrier')

In [17]:
# filter only qrels that are annotated by human legal experts
df_qrels_human = df_qrels.query("source=='human'")
df_qrels_human

,qid,docno,label,source
38,62015CJ0356-107,62009CJ0269-72,1,human
40,62015CJ0356-107,62010CJ0383-53,1,human
42,62015CJ0356-110,61991CJ0102-52,1,human
44,62015CJ0356-110,61997CJ0202-32,1,human
45,62015CJ0356-25,62009CJ0376-32,1,human
...,...,...,...,...
14446,62019CJ0711-25,62014CJ0336-70,1,human
14447,62019CJ0711-25,62014CJ0613-66,1,human
14448,62019CJ0711-25,62015CJ0303-18,1,human
14449,62019CJ0711-25,62016CJ0144-25,1,human


In [18]:
bm25 = index.bm25(num_results=100) # consider only top 100 results

df_result, df_per_q = pt.Experiment(
    [bm25],
    df_queries,
    df_qrels_human,
    names=["BM25"],
    eval_metrics=["num_q", "num_rel", "success", "recip_rank"],
    perquery="both",
    save_dir=save_folder,
    filter_by_qrels=True #this changes the default behaviour; only intersection between test_queries and qrel are used to evaluate
)

In [19]:
df_result

,name,num_q,num_rel,recip_rank,Success@1,Success@5,Success@10
0,BM25,69.0,409.0,0.774638,0.695652,0.869565,0.942029


In [20]:
df_per_q.head(10)

,name,qid,measure,value
0,BM25,62015CJ0356-107,num_q,1.0
1,BM25,62015CJ0356-107,num_rel,2.0
2,BM25,62015CJ0356-107,recip_rank,1.0
3,BM25,62015CJ0356-107,Success@1,1.0
4,BM25,62015CJ0356-107,Success@5,1.0
5,BM25,62015CJ0356-107,Success@10,1.0
6,BM25,62015CJ0356-110,num_q,1.0
7,BM25,62015CJ0356-110,num_rel,2.0
8,BM25,62015CJ0356-110,recip_rank,1.0
9,BM25,62015CJ0356-110,Success@1,1.0


## Reranking with MonoT5 (cross-encoder)

MonoT5 is a cross-encoder semantic re-ranker that can be used to refine the first-stage retrieval (but it can also be used directly as a first-stage dense retriever, if computational resources allow). This part is expected to be slow without GPU

In [ ]:
# !pip install --upgrade git+https://github.com/terrierteam/pyterrier_t5.git --user

In [21]:
from pyterrier_t5 import MonoT5ReRanker 

monoT5 = MonoT5ReRanker()

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

The index files in HuggingFace do not have the text, so we need to load them separately

In [28]:
doc_config = f"doc_{doc}"

docs = load_dataset("theresiavr/legalpincite", doc_config, split=split)
df_docs = docs.to_pandas()

In [65]:
import numpy as np

def get_text(row):
    docno = row["docno"]
    try: 
        return df_docs.loc[df_docs.docno==docno, "text"].values[0]
        
    except:
        return ""

In [66]:
doc_text = pt.apply.text(get_text)
pipeline = bm25 >> doc_text >> monoT5 # remember that bm25 is already cut at 100

In [68]:
pt.Experiment(
    [bm25, pipeline],
    df_queries,
    df_qrels_human.sample(10, random_state=50), #take randomly, since this is an example; in real eval the sampling should be omitted
    names=["BM25", "BM25+MonoT5"],
    eval_metrics=["num_q", "num_rel", "success", "recip_rank"],
    filter_by_qrels=True #this changes the default behaviour; only intersection between test_queries and qrel are used to evaluate
)

monoT5:   0%|          | 0/250 [00:00<?, ?batches/s]

,name,num_q,num_rel,recip_rank,Success@1,Success@5,Success@10
0,BM25,10.0,10.0,0.337714,0.2,0.4,0.6
1,BM25+MonoT5,10.0,10.0,0.282569,0.2,0.3,0.7


Here, reranking with MonoT5 improves Success@10

## Reranking with LegalBERT (bi-encoder)

Modified from: https://pyterrier.readthedocs.io/en/latest/text.html#examples-of-sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

bimodel = SentenceTransformer('nlpaueb/bert-base-uncased-eurlex')

def _biencoder_apply(df : pd.DataFrame):
    from sentence_transformers.util import cos_sim
    query_embs = bimodel.encode(df['query'].values)
    doc_embs = bimodel.encode(df['text'].values)
    scores =  cos_sim(query_embs, doc_embs)
    return scores[0]

In [73]:
bi_encT = pt.apply.doc_score(_biencoder_apply, batch_size=4)

In [77]:
pt.Experiment(
    [bm25, bm25 >> doc_text >> bi_encT],
    df_queries,
    df_qrels_human.sample(10, random_state=50), #sample to demo on a small subset
    names=["BM25", "BM25+LegalBert"],
    eval_metrics=["num_q", "num_rel", "success", "recip_rank"],
    filter_by_qrels=True 
)

,name,num_q,num_rel,recip_rank,Success@1,Success@5,Success@10
0,BM25,10.0,10.0,0.337714,0.2,0.4,0.6
1,BM25+LegalBert,10.0,10.0,0.172560,0.1,0.2,0.4
